In [ ]:
import h5py
import numpy as np
import os

def print_hdf5_attributes(hdf5_path):
    """
    Print attributes of all datasets in an HDF5 file, specifically calculating
    mean and std for the 'eeg_data' field in each dataset, using robust access methods.
    
    Parameters:
    hdf5_path (str): Path to the HDF5 file
    
    Output includes for each dataset:
    - Dataset path
    - Chunk size (if applicable)
    - Shape of the dataset
    - Total number of elements
    - Mean and standard deviation of the 'eeg_data' field for the first 1000 samples
    """
    if not os.path.exists(hdf5_path):
        print(f"Error: File not found at {hdf5_path}")
        return
    
    try:
        with h5py.File(hdf5_path, 'r') as f:
            dataset_paths = []
            
            def collect_datasets(group, prefix=''):
                for name in group:
                    try:
                        item = group[name]
                        path = f"{prefix}/{name}" if prefix else name
                        if isinstance(item, h5py.Dataset):
                            dataset_paths.append(path)
                        elif isinstance(item, h5py.Group):
                            collect_datasets(item, path)
                    except Exception as e:
                        print(f"Warning: Could not access {name} in group {prefix}: {str(e)}")
            
            try:
                collect_datasets(f)
            except Exception as e:
                print(f"Error during manual traversal: {str(e)}")
                return
            
            if not dataset_paths:
                print("No datasets found in the file.")
                return
            
            print(f"Found {len(dataset_paths)} datasets")
            
            for i, path in enumerate(dataset_paths):
                print(f"\nProcessing dataset {i+1}/{len(dataset_paths)}: {path}")
                try:
                    ds = f[path]
                    
                    print(f"  Chunk size: {ds.chunks if ds.chunks else 'N/A'}")
                    print(f"  Shape: {ds.shape}")
                    print(f"  Total elements: {ds.size}")
                    print(f"  Data type: {ds.dtype}")
                    

                    if hasattr(ds.dtype, 'names') and 'eeg_data' in ds.dtype.names:
                        sample_size = min(2000, ds.size)
                        
                        if sample_size == 0:
                            print("  Dataset is empty")
                            continue
                        
                        try:

                            samples = ds[:sample_size]
                            
                            eeg_data = samples['eeg_data'].astype(np.float32)
                            
                            mean = np.mean(eeg_data)
                            std = np.std(eeg_data)
                            
                            print(f"  EEG data shape: {eeg_data.shape}")
                            print(f"  Mean of eeg_data (first {sample_size} samples): {mean:.6f}")
                            print(f"  Std of eeg_data (first {sample_size} samples): {std:.6f}")
                            
                        except Exception as read_err:
                            print(f"  Error reading data: {str(read_err)}")
                    else:
                        print("  Dataset does not contain 'eeg_data' field")
                
                except Exception as ds_err:
                    print(f"  Error accessing dataset: {str(ds_err)}")
    
    except IOError as e:
        print(f"Error opening HDF5 file: {str(e)}")
    except Exception as e:
        print(f"Unexpected error: {str(e)}")


hdf5_path = os.path.join(os.getcwd(), 'data_label.h5')
print(f"Analyzing HDF5 file: {hdf5_path}")
print_hdf5_attributes(hdf5_path)

In [ ]:
from EEG_Encoder.Tools.dataBuilder import CLIPDataset
dataset_name='thingEEG'
ds=CLIPDataset(hdf5_file_path='data_label.h5',
                mode='test',
                dataset_name=dataset_name,
                ground_truth_dir='CLIP_groundTruth')
sample=ds[0]
print(sample.keys())
print(sample['eeg_data'].shape)
print(sample['img_features'][0])
print(sample['text_features'][0])
print(sample['text'])
print(sample['img_path'])

In [ ]:
import h5py
import os

def delete_hdf5_dataset(hdf5_path, dataset_name):
    """
    Deletes a specific dataset from an HDF5 file.
    
    This function will:
    1. Verify the HDF5 file exists
    2. Open the file in read-write mode
    3. Check if the specified dataset exists
    4. Delete the dataset if found
    5. Handle errors and edge cases appropriately
    
    Parameters:
    hdf5_path (str): Path to the HDF5 file
    dataset_name (str): Full path to the dataset to be deleted (e.g., '/group/subgroup/dataset_name')
    
    Returns:
    None: Prints status messages about the operation
    """
    
    # Check if file exists
    if not os.path.exists(hdf5_path):
        print(f"Error: File not found at {hdf5_path}")
        return
    
    try:
        # Open file in read-write mode
        with h5py.File(hdf5_path, 'a') as f:
            # Check if dataset exists
            if dataset_name in f:
                # Verify it's actually a dataset (not a group)
                if isinstance(f[dataset_name], h5py.Dataset):
                    # Delete the dataset
                    del f[dataset_name]
                    print(f"Successfully deleted dataset: {dataset_name}")
                else:
                    print(f"Error: '{dataset_name}' is a group, not a dataset. Deletion aborted.")
            else:
                print(f"Dataset '{dataset_name}' not found in file.")
    
    except IOError as e:
        print(f"Error opening HDF5 file: {str(e)}")
    except Exception as e:
        print(f"Unexpected error: {str(e)}")



delete_hdf5_dataset(hdf5_path, 'xxxxx')